# Групповой анализ

## Настройка окружения


In [ ]:
!gdown --folder 1AZdHTmaW_d-fHn2kzxyGkvspYpweznxv --remaining-ok

Retrieving folder contents
Retrieving folder 1dbCaaYkhxbm0QCsWVavgqxp7kyr4ybL_ bombus_affinis
Processing file 15EEch8Kb6R0V0HP9D-OVzz1fhMXzdrLq annotation.gff
Processing file 1gZeIxivMPKwaY6dKJc1Z_4HCfgQmCC6k g4.bed
Processing file 1RLaU55cqvoLoGxHGaRJhzbTmSeeS3JS1 genome.fna
Processing file 1uBwnynGqpm8BiEXxH8uaTkgyih6PM6zZ proteins.faa
Processing file 1RpLpZwoRUAhCBRJPL9AD2qcKS7lv21ib zdna.bed
Processing file 1gXZjoiVi-gqosG8S62RUXJr3KwX19Dvw zdnabert.bed
Retrieving folder 18P_ZHkgoEux1Mmz5WoSpEdkx0zKxj069 bombus_fervidus
Processing file 1C-TwYhToA9Hwac0AnPBHemur4jrQomVw annotation.gff
Processing file 1zlF8tmy1kpJh4KqMaZqYCvZyQEdUNzsr g4.bed
Processing file 1vXs09nbQYyp8Hjx-rIjP8LHA1sh1ss2A genome.fna
Processing file 10KyCl3fcgwQxiCmUxxcNiZ_2IhlJRxAX proteins.faa
Processing file 11oF2ZuXoE2bTJ2N6whw0dZDzuVVX5lPI zdna.bed
Processing file 19-PAU8kIhcNMDhPEGEp-b_Jfc0FCKJpr zdnabert.bed
Retrieving folder 13YZJNFctpOrZw5rmdlCKDe2_daBdGOg5 bombus_huntii
Processing file 1tTHQux62lfQgXDLs4qX

In [ ]:
!wget https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh

--2026-06-11 17:59:55--  https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-Linux-x86_64.sh [following]
--2026-06-11 17:59:55--  https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-Linux-x86_64.sh
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/221584272/dd517a4f-612c-449e-89ad-3755d98f8038?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-11T18%3A46%3A32Z&rscd=attachment%3B+filename%3DMiniforge3-Linux-x86_64.sh&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt

In [ ]:
!chmod u+x Miniforge3-Linux-x86_64.sh

In [ ]:
!bash ./Miniforge3-Linux-x86_64.sh -b -p "${HOME}/miniforge3"

ERROR: File or directory already exists: '/root/miniforge3'
If you want to update an existing installation, use the -u option.


In [ ]:
!${HOME}/miniforge3/bin/conda init bash

no change     /root/miniforge3/condabin/conda
no change     /root/miniforge3/bin/conda
no change     /root/miniforge3/bin/activate
no change     /root/miniforge3/bin/deactivate
no change     /root/miniforge3/etc/profile.d/conda.sh
no change     /root/miniforge3/etc/fish/conf.d/conda.fish
no change     /root/miniforge3/shell/condabin/Conda.psm1
no change     /root/miniforge3/shell/condabin/conda-hook.ps1
no change     /root/miniforge3/lib/python3.13/site-packages/xontrib/conda.xsh
no change     /root/miniforge3/etc/profile.d/conda.csh
no change     /root/.bashrc
No action taken.


In [ ]:
%%bash
${HOME}/miniforge3/bin/conda install -y -c bioconda -c conda-forge \
    mafft fasttree diamond bedtools samtools biopython pandas numpy requests 2>&1 | tail -5

Solving environment: / - \ done

# All requested packages already installed.



In [ ]:
!pip install biopython

## Конфигурация

тут опять нужно как-то извратиться, чтобы весь тулинг стал доступен без дублирования miniforge каждый раз

In [ ]:
import os, re, subprocess, shutil, glob
import pandas as pd
import numpy as np
import requests
from Bio import SeqIO

def _ensure_on_path(tools=('mafft','diamond','bedtools','FastTree','fasttree')):
    cand = ([os.path.expanduser('~/miniforge3/bin')]
            + sorted(glob.glob(os.path.expanduser('~/miniforge3/envs/*/bin')))
            + sorted(glob.glob(os.path.expanduser('~/miniconda3/envs/*/bin')))
            + sorted(glob.glob(os.path.expanduser('~/anaconda3/envs/*/bin'))))
    for t in tools:
        if shutil.which(t) is None:
            for d in cand:
                if os.path.exists(os.path.join(d, t)):
                    os.environ['PATH'] = d + os.pathsep + os.environ['PATH']; break
_ensure_on_path()
FASTTREE = 'FastTree' if shutil.which('FastTree') else 'fasttree'

ORGANISMS = {
    'bombus_terrestris': {
        'proteome':'bioinformatics-project/bombus_terrestris/proteins.faa',
        'genome'  :'bioinformatics-project/bombus_terrestris/genome.fna',
        'gff'     :'bioinformatics-project/bombus_terrestris/annotation.gff',
        'g4'      :'bioinformatics-project/bombus_terrestris/g4.bed',
        'zdna'    :'bioinformatics-project/bombus_terrestris/zdna.bed',
    },
    'bombus_pascuorum': {
        'proteome':'bioinformatics-project/bombus_pascuorum/proteins.faa',
        'genome'  :'bioinformatics-project/bombus_pascuorum/genome.fna',
        'gff'     :'bioinformatics-project/bombus_pascuorum/annotation.gff',
        'g4'      :'bioinformatics-project/bombus_pascuorum/g4.bed',
        'zdna'    :'bioinformatics-project/bombus_pascuorum/zdna.bed',
    },'bombus_vancouverensis': {
        'proteome':'bioinformatics-project/bombus_vancouverensis/proteins.faa',
        'genome'  :'bioinformatics-project/bombus_vancouverensis/genome.fna',
        'gff'     :'bioinformatics-project/bombus_vancouverensis/annotation.gff',
        'g4'      :'bioinformatics-project/bombus_vancouverensis/g4.bed',
        'zdna'    :'bioinformatics-project/bombus_vancouverensis/zdna.bed',
    }, 'bombus_huntii': {
        'proteome':'bioinformatics-project/bombus_huntii/proteins.faa',
        'genome'  :'bioinformatics-project/bombus_huntii/genome.fna',
        'gff'     :'bioinformatics-project/bombus_huntii/annotation.gff',
        'g4'      :'bioinformatics-project/bombus_huntii/g4.bed',
        'zdna'    :'bioinformatics-project/bombus_huntii/zdna.bed',
    }, 'bombus_vosnesenskii': {            # scaffold-level assembly (no chromosomes)
        'proteome':'bioinformatics-project/bombus_vosnesenskii/proteins.faa',
        'genome'  :'bioinformatics-project/bombus_vosnesenskii/genome.fna',
        'gff'     :'bioinformatics-project/bombus_vosnesenskii/annotation.gff',
        'g4'      :'bioinformatics-project/bombus_vosnesenskii/g4.bed',
        'zdna'    :'bioinformatics-project/bombus_vosnesenskii/zdna.bed',
    },
    'bombus_fervidus': {
        'proteome':'bioinformatics-project/bombus_fervidus/proteins.faa',
        'genome'  :'bioinformatics-project/bombus_fervidus/genome.fna',
        'gff'     :'bioinformatics-project/bombus_fervidus/annotation.gff',
        'g4'      :'bioinformatics-project/bombus_fervidus/g4.bed',
        'zdna'    :'bioinformatics-project/bombus_fervidus/zdna.bed',
    }, 'bombus_affinis': {
        'proteome':'bioinformatics-project/bombus_affinis/proteins.faa',
        'genome'  :'bioinformatics-project/bombus_affinis/genome.fna',
        'gff'     :'bioinformatics-project/bombus_affinis/annotation.gff',
        'g4'      :'bioinformatics-project/bombus_affinis/g4.bed',
        'zdna'    :'bioinformatics-project/bombus_affinis/zdna.bed',
    }, 'bombus_pyrosoma': {
        'proteome':'bioinformatics-project/bombus_pyrosoma/proteins.faa',
        'genome'  :'bioinformatics-project/bombus_pyrosoma/genome.fna',
        'gff'     :'bioinformatics-project/bombus_pyrosoma/annotation.gff',
        'g4'      :'bioinformatics-project/bombus_pyrosoma/g4.bed',
        'zdna'    :'bioinformatics-project/bombus_pyrosoma/zdna.bed',
    },
}

REFERENCES = {
    # CTCF + cohesin complex
    'CTCF' : ['P49711'],
    'SMC1A': ['Q14683'],
    'SMC3' : ['Q9UQE7'],
    'RAD21': ['O60216'],
    'STAG2': ['Q8N3U4'],
    'NIPBL': ['Q6KC79'],
    # something else
    'DNMT1': ['P26358'],
    'HDAC1': ['Q13547'],
    'EZH2' : ['Q15910'],
}

PROM, EVALUE = 1000, 1e-5
for d in ['group/db','group/blast','group/seqs','group/aln','group/trees',
          'group/promoters','group/refs']:
    os.makedirs(d, exist_ok=True)

print('Registry check:')
for org, p in ORGANISMS.items():
    for k, v in p.items():
        print(f'  {org}/{k}: {"OK" if os.path.exists(v) else "MISSING -> "+v}')
print('FastTree:', FASTTREE)

Registry check:
  bombus_terrestris/proteome: OK
  bombus_terrestris/genome: OK
  bombus_terrestris/gff: OK
  bombus_terrestris/g4: OK
  bombus_terrestris/zdna: OK
  bombus_pascuorum/proteome: OK
  bombus_pascuorum/genome: OK
  bombus_pascuorum/gff: OK
  bombus_pascuorum/g4: OK
  bombus_pascuorum/zdna: OK
  bombus_vancouverensis/proteome: OK
  bombus_vancouverensis/genome: OK
  bombus_vancouverensis/gff: OK
  bombus_vancouverensis/g4: OK
  bombus_vancouverensis/zdna: OK
  bombus_huntii/proteome: OK
  bombus_huntii/genome: OK
  bombus_huntii/gff: OK
  bombus_huntii/g4: OK
  bombus_huntii/zdna: OK
  bombus_vosnesenskii/proteome: OK
  bombus_vosnesenskii/genome: OK
  bombus_vosnesenskii/gff: OK
  bombus_vosnesenskii/g4: OK
  bombus_vosnesenskii/zdna: OK
  bombus_fervidus/proteome: OK
  bombus_fervidus/genome: OK
  bombus_fervidus/gff: OK
  bombus_fervidus/g4: OK
  bombus_fervidus/zdna: OK
  bombus_affinis/proteome: OK
  bombus_affinis/genome: OK
  bombus_affinis/gff: OK
  bombus_affinis/g

## 3. Helper functions

In [ ]:
def sh(cmd): subprocess.run(cmd, shell=True, check=True)

def fetch_uniprot(acc):
    r = requests.get(f'https://rest.uniprot.org/uniprotkb/{acc}.fasta', timeout=60)
    r.raise_for_status(); return r.text

_CACHE = {}
def proteome(org):
    k = 'prot_' + org
    if k not in _CACHE: _CACHE[k] = SeqIO.index(ORGANISMS[org]['proteome'], 'fasta')
    return _CACHE[k]

def chrom_sizes(org):
    k = 'sz_' + org
    if k not in _CACHE:
        _CACHE[k] = {r.id: len(r.seq) for r in SeqIO.parse(ORGANISMS[org]['genome'], 'fasta')}
    return _CACHE[k]

def _gff_maps(org):
    k = 'gff_' + org
    if k in _CACHE: return _CACHE[k]
    gene_loc, rna2gene, rna2prot = {}, {}, {}
    with open(ORGANISMS[org]['gff']) as fh:
        for line in fh:
            if line.startswith('#'): continue
            f = line.rstrip('\n').split('\t')
            if len(f) < 9: continue
            a = dict(x.split('=',1) for x in f[8].split(';') if '=' in x)
            if   f[2] == 'gene': gene_loc[a.get('ID')] = (f[0], int(f[3]), int(f[4]), f[6])
            elif f[2] in ('mRNA','transcript') and 'ID' in a: rna2gene[a['ID']] = a.get('Parent')
            elif f[2] == 'CDS' and 'protein_id' in a: rna2prot.setdefault(a.get('Parent'), a['protein_id'])
    prot2gene = {rna2prot[r]: rna2gene[r] for r in rna2prot if r in rna2gene}
    _CACHE[k] = (gene_loc, rna2gene, rna2prot, prot2gene); return _CACHE[k]

def _safe_intersect(a_bed, b_bed, extra='-u'):
    if not (a_bed and os.path.exists(a_bed) and os.path.exists(b_bed)): return []
    try:
        return subprocess.check_output(f'bedtools intersect -a {a_bed} -b {b_bed} {extra}',
                                       shell=True, stderr=subprocess.DEVNULL).decode().splitlines()
    except subprocess.CalledProcessError:
        return []

def diamond_best(query_faa, org, tag='ref'):
    db, out = f'group/db/{org}', f'group/blast/{org}_{tag}.tsv'
    if not os.path.exists(db + '.dmnd'):
        sh(f'diamond makedb --in {ORGANISMS[org]["proteome"]} -d {db} --quiet')
    sh(f'diamond blastp -q {query_faa} -d {db} -o {out} --very-sensitive '
       f'--max-target-seqs 1 --evalue {EVALUE} --threads 4 '
       f'--outfmt 6 qseqid sseqid pident evalue bitscore --quiet')
    best = {}
    for line in open(out):
        q, s, pid, ev, bs = line.rstrip('\n').split('\t'); ev, bs = float(ev), float(bs)
        if q not in best or bs > best[q][3]: best[q] = (s, float(pid), ev, bs)
    return {q: (s, pid, ev) for q, (s, pid, ev, bs) in best.items()}

def mafft(in_faa, out_aln):
    sh(f'mafft --auto --quiet {in_faa} > {out_aln}')

def fasttree(aln, tree, nt=False):
    sh(f'{FASTTREE} {"-nt -gtr" if nt else "-lg"} -quiet {aln} > {tree} 2>/dev/null')

def gene_locus(org, protein_id):
    gene_loc, _, _, prot2gene = _gff_maps(org)
    return gene_loc.get(prot2gene.get(protein_id))

def promoter_bed(org, locus):
    chrom, s, e, strand = locus; s -= 1
    clen = chrom_sizes(org).get(chrom, e + PROM)
    ps, pe = (max(0, s - PROM), s) if strand == '+' else (e, min(clen, e + PROM))
    return f'{chrom}\t{ps}\t{pe}\tprom\t0\t{strand}\n' if pe > ps else None

def getfasta(org, bed_lines):
    if not bed_lines or not os.path.exists(ORGANISMS[org]['genome']): return []
    tmp = 'group/promoters/_q.bed'
    with open(tmp, 'w') as fh: fh.writelines(bed_lines)
    flag = '-s' if all(len(l.rstrip('\n').split('\t')) >= 6 for l in bed_lines) else ''
    try:
        out = subprocess.check_output(
            f'bedtools getfasta -fi {ORGANISMS[org]["genome"]} -bed {tmp} {flag}',
            shell=True, stderr=subprocess.DEVNULL).decode()
    except subprocess.CalledProcessError:
        return []
    recs, name = [], None
    for ln in out.splitlines():
        if ln.startswith('>'): name = ln[1:]
        elif name is not None: recs.append((name, ln)); name = None
    return recs

def promoter_structures(org, protein_id, struct_key):
    loc = gene_locus(org, protein_id)
    if not loc: return None, []
    pline = promoter_bed(org, loc)
    if not pline: return None, []
    pf = 'group/promoters/_prom.bed'
    with open(pf, 'w') as fh: fh.write(pline)
    inter = _safe_intersect(ORGANISMS[org].get(struct_key), pf, '-u')
    prom = getfasta(org, [pline])
    sts  = getfasta(org, [l + '\n' for l in inter]) if inter else []
    return (prom[0][1] if prom else None), [s for _, s in sts]

def build_gene_promoters(org):
    gene_loc, rna2gene, rna2prot, _ = _gff_maps(org)
    out = f'group/promoters/{org}_gene_prom.bed'; cs = chrom_sizes(org); seen = set()
    with open(out, 'w') as o:
        for rna, gene in rna2gene.items():
            if gene in seen or rna not in rna2prot or gene not in gene_loc: continue
            seen.add(gene); chrom, s, e, strand = gene_loc[gene]; s -= 1
            clen = cs.get(chrom, e + PROM)
            ps, pe = (max(0, s - PROM), s) if strand == '+' else (e, min(clen, e + PROM))
            if pe > ps: o.write(f'{chrom}\t{ps}\t{pe}\t{rna2prot[rna]}\t0\t{strand}\n')
    return out

def promoter_structure_proteins(org, struct_key, limit=40):
    pb = build_gene_promoters(org)
    prots = [l.split('\t')[3] for l in _safe_intersect(pb, ORGANISMS[org].get(struct_key), '-u')]
    return list(dict.fromkeys(prots))[:limit]


In [ ]:
import bisect, csv as _csv
FLANK = 60

def _structure_pieces(org, protein_id, struct_key, flank=FLANK):
    loc = gene_locus(org, protein_id)
    if not loc:
        return []
    pline = promoter_bed(org, loc)
    if not pline:
        return []
    with open('group/promoters/_prom.bed', 'w') as fh:
        fh.write(pline)
    inter = _safe_intersect(ORGANISMS[org].get(struct_key), 'group/promoters/_prom.bed', '-u')
    if not inter:
        return []
    clen = chrom_sizes(org)
    reg_lines, ctx_lines, meta = [], [], []
    for l in inter:
        f = l.rstrip('\n').split('\t')
        chrom, s, e = f[0], int(f[1]), int(f[2])
        strand = f[5] if len(f) >= 6 else '+'
        L = clen.get(chrom, e + flank)
        cs, ce = max(0, s - flank), min(L, e + flank)
        reg_lines.append(f'{chrom}\t{s}\t{e}\tr\t0\t{strand}\n')
        ctx_lines.append(f'{chrom}\t{cs}\t{ce}\tc\t0\t{strand}\n')
        meta.append((chrom, s, e, strand))
    rseq, cseq = getfasta(org, reg_lines), getfasta(org, ctx_lines)
    if len(rseq) != len(meta) or len(cseq) != len(meta):
        return []
    return [{'chrom': c, 's': s, 'e': e, 'strand': st, 'region': rseq[i][1], 'context': cseq[i][1]}
            for i, (c, s, e, st) in enumerate(meta)]

def _feat_index(org):
    k = 'featidx_' + org
    if k in _CACHE:
        return _CACHE[k]
    genes, exons = {}, {}
    with open(ORGANISMS[org]['gff']) as fh:
        for line in fh:
            if line.startswith('#'):
                continue
            f = line.split('\t')
            if len(f) < 9:
                continue
            if f[2] == 'gene':
                genes.setdefault(f[0], []).append((int(f[3]) - 1, int(f[4])))
            elif f[2] == 'exon':
                exons.setdefault(f[0], []).append((int(f[3]) - 1, int(f[4])))
    for d in (genes, exons):
        for c in d:
            d[c].sort()
    _CACHE[k] = (genes, exons)
    return _CACHE[k]

def _overlaps(intervals, s, e):
    if not intervals:
        return False
    hi = bisect.bisect_left([iv[0] for iv in intervals], e)
    return any(intervals[k][1] > s for k in range(hi))

def feature_of(org, chrom, s, e):
    genes, exons = _feat_index(org)
    if _overlaps(exons.get(chrom, []), s, e):
        return 'exon'
    if _overlaps(genes.get(chrom, []), s, e):
        return 'intron'
    return 'intergenic'

## Поиск генов в протеоме


Используем diamond (аналог bastp), отсекаем по e-value 1e-5.

In [ ]:
ref_faa = 'group/refs/references.faa'
ref_seqs = {}
with open(ref_faa, 'w') as out:
    for gene, accs in REFERENCES.items():
        ref_seqs[gene] = []
        for acc in accs:
            try:
                seq = ''.join(fetch_uniprot(acc).split('\n')[1:])
            except Exception as e:
                print(f'  fetch failed {gene}/{acc}: {e}'); continue
            ref_seqs[gene].append((acc, seq)); out.write(f'>{gene}__{acc}\n{seq}\n')

ortholog, rows = {}, []
for org in ORGANISMS:
    best = diamond_best(ref_faa, org, tag='ref')
    row = {'organism': org}
    for gene in REFERENCES:
        cand = [best[q] for q in best if q.split('__')[0] == gene]
        b = min(cand, key=lambda x: x[2]) if cand else None
        ortholog.setdefault(gene, {})[org] = b
        row[gene] = f'{b[1]:.0f}% ({b[2]:.0e})' if b else '— absent'
    rows.append(row)

presence = pd.DataFrame(rows).set_index('organism')
presence.to_csv('group/ctcf_cohesin_presence.csv')
print('=== CTCF / cohesin presence — best BLASTP hit: %identity (E-value) ===')
print(presence.to_string())

=== CTCF / cohesin presence — best BLASTP hit: %identity (E-value) ===
                           CTCF        SMC1A         SMC3         RAD21        STAG2        NIPBL        DNMT1         HDAC1          EZH2
organism                                                                                                                                  
bombus_terrestris      — absent  49% (0e+00)  52% (0e+00)  39% (7e-118)  52% (0e+00)  50% (0e+00)  56% (0e+00)  81% (3e-276)  58% (2e-262)
bombus_pascuorum       — absent  49% (0e+00)  52% (0e+00)  38% (2e-117)  52% (0e+00)  49% (0e+00)  55% (0e+00)  81% (3e-276)  57% (7e-263)
bombus_vancouverensis  — absent  49% (0e+00)  52% (0e+00)  38% (2e-117)  52% (0e+00)  49% (0e+00)  55% (0e+00)  81% (4e-277)  58% (8e-263)
bombus_huntii          — absent  49% (0e+00)  52% (0e+00)  38% (3e-117)  52% (0e+00)  49% (0e+00)  56% (0e+00)  81% (4e-276)  58% (9e-263)
bombus_vosnesenskii    — absent  49% (0e+00)  52% (0e+00)  38% (2e-117)  52% (0e+00)  49% (0e+0

## Выравнивание и построение филогенетических деревьев

Строим множественное выравнивание при помощи MAFFT, используем параметр `--auto`, чтобы программа сама выбрала оптимальный алгоритм и параметры. Для построения деревьев используем утилиту FastTree.

In [ ]:
from Bio import Phylo

OUTGROUP_TAG = '_ref_'

for gene in REFERENCES:
    recs = [(f'{gene}_ref_{acc}', seq) for acc, seq in ref_seqs.get(gene, [])]
    for org, hit in ortholog.get(gene, {}).items():
        if not hit: continue
        rec = proteome(org).get(hit[0])
        if rec: recs.append((org, str(rec.seq)))
    if len(recs) < 2:
        print(f'{gene}: {len(recs)} seq(s) — skipped'); continue
    faa, aln, tre = f'group/seqs/{gene}.faa', f'group/aln/{gene}.aln', f'group/trees/{gene}.nwk'
    with open(faa, 'w') as fh:
        for n, s in recs: fh.write(f'>{n}\n{s}\n')
    mafft(faa, aln)
    if len(recs) >= 3:
        fasttree(aln, tre)
        tree = Phylo.read(tre, 'newick')
        og = [t for t in tree.get_terminals() if OUTGROUP_TAG in t.name]
        if og:
            try:
                tree.root_with_outgroup(*og)
            except Exception as e:
                print(f'  {gene}: rooting failed ({e}); left unrooted')
        else:
            print(f'  {gene}: no reference tip found — left unrooted')
        Phylo.write(tree, tre, 'newick')
        rooted_on = ', '.join(t.name for t in og) or '—'
        print(f'\n=== {gene} ({len(recs)} taxa, rooted on outgroup: {rooted_on}) ===')
        Phylo.draw_ascii(tree)
    else:
        print(f'{gene}: aligned {len(recs)} seqs -> {aln}')
print('\nAlignments -> group/aln/ , rooted trees -> group/trees/ '
      '(outgroup = human reference; view .nwk in iTOL/FigTree)')

CTCF: 1 seq(s) — skipped

=== SMC1A (9 taxa, rooted on outgroup: SMC1A_ref_Q14683) ===
                                                      , bombus_vancouverensis
                                                      |
                                                      | bombus_huntii
                                                      |
                                                      | bombus_vosnesenskii
                                                      |
                                                      | bombus_pyrosoma
                                                      |
                                                      , bombus_fervidus
                                                      |
  ____________________________________________________, bombus_terrestris
 |                                                    |
 |                                                    | bombus_affinis
_|                                                    |
 |      

## Гены с G-квадруплексами в промотерах

Как написано в задании, промотером мы считаем кусочек 1kb upstream от TSS, вытаскиваем оттуда квадруплекс и выравниваем между организмами. Далее разбиваем квадруплекс на петли.

In [ ]:
REF_ORG = next(iter(ORGANISMS))

def g_profile(seq):
    tracts = [len(m.group()) for m in re.finditer(r'G{3,5}|C{3,5}', seq, re.IGNORECASE)]
    loops  = [len(x) for x in re.split(r'G{3,5}|C{3,5}', seq, flags=re.IGNORECASE) if x]
    return len(tracts), tracts, loops

def orthologous_promoter_genes(struct_key, label, limit=200):
    cand = promoter_structure_proteins(REF_ORG, struct_key, limit)
    print(f'{REF_ORG}: {len(cand)} candidate genes with a promoter {label}')
    if not cand: return {}
    qf = f'group/seqs/{label}_query.faa'; idx = proteome(REF_ORG)
    with open(qf, 'w') as fh:
        for p in cand:
            if p in idx: fh.write(f'>{p}\n{idx[p].seq}\n')
    omap = {p: {REF_ORG: p} for p in cand}
    for org in ORGANISMS:
        if org == REF_ORG: continue
        best = diamond_best(qf, org, tag=label)
        nfound = 0
        for p in cand:
            if best.get(p): omap[p][org] = best[p][0]; nfound += 1
        print(f'  {org}: orthologs found for {nfound}/{len(cand)} candidates')
    return omap

from collections import Counter
omap = orthologous_promoter_genes('g4', 'G4')

per_org, allseqs = Counter(), {}
for p, m in omap.items():
    pieces = {}
    for org, prot in m.items():
        pcs = _structure_pieces(org, prot, 'g4')
        if pcs:
            pieces[org] = max(pcs, key=lambda d: len(d['region']))
            per_org[org] += 1
    allseqs[p] = pieces
print('promoter-G4 recovered per organism:', dict(per_org))

shared = [(p, s) for p, s in allseqs.items() if len(s) >= 2]
print(f'{len(shared)} gene(s) have a promoter G4 in >=2 organisms')

bed_rows, feat_rows = [], []
for i, (p, pieces) in enumerate(shared[:9], 1):
    print(f'\n=== promoter-G4 ortholog #{i}  (ref protein {p}) ===')
    for org, d in pieces.items():
        n, tr, lp = g_profile(d['region'])
        loc = feature_of(org, d['chrom'], d['s'], d['e'])
        print(f'  {org:20s} len={len(d["region"]):3d} tracts={n} {tr} loops={lp}  '
              f'[{d["chrom"]}:{d["s"]}-{d["e"]} {d["strand"]} -> {loc}]')
    # G4-кусочек
    faa, aln = f'group/seqs/promG4_{i}.fa', f'group/aln/promG4_{i}.aln'
    with open(faa, 'w') as fh:
        for org, d in pieces.items(): fh.write(f'>{org}\n{d["region"]}\n')
    mafft(faa, aln)
    for r in SeqIO.parse(aln, 'fasta'): print(f'    region  {r.id:20s} {r.seq}')
    # G4-кусочек +- 60 нуклеотидов
    cfaa, caln = f'group/seqs/promG4_{i}_context.fa', f'group/aln/promG4_{i}_context.aln'
    with open(cfaa, 'w') as fh:
        for org, d in pieces.items(): fh.write(f'>{org}\n{d["context"]}\n')
    mafft(cfaa, caln)
    print(f'    context -> {caln}')
    # BED-файл
    for org, d in pieces.items():
        loc = feature_of(org, d['chrom'], d['s'], d['e'])
        bed_rows.append((d['chrom'], d['s'], d['e'], f'promG4_{i}|{org}|{loc}', 0, d['strand']))
        feat_rows.append({'alignment': f'promG4_{i}', 'organism': org, 'chrom': d['chrom'],
                          'start': d['s'], 'end': d['e'], 'strand': d['strand'],
                          'region_len': len(d['region']), 'feature': loc})

with open('group/aln/promG4_pieces.bed', 'w') as fh:
    for r in sorted(bed_rows): fh.write('\t'.join(map(str, r)) + '\n')
with open('group/aln/promG4_pieces_features.csv', 'w', newline='') as fh:
    w = _csv.DictWriter(fh, fieldnames=['alignment', 'organism', 'chrom', 'start', 'end',
                                        'strand', 'region_len', 'feature'])
    w.writeheader(); w.writerows(feat_rows)

print('\npromoter-G4 region alignments  -> group/aln/promG4_*.aln')
print('promoter-G4 context alignments -> group/aln/promG4_*_context.aln')
print('aligned pieces (coordinates)   -> group/aln/promG4_pieces.bed')
print('feature labels per piece       -> group/aln/promG4_pieces_features.csv')
print('feature distribution of aligned G4 pieces:', dict(Counter(r['feature'] for r in feat_rows)))

bombus_terrestris: 120 candidate genes with a promoter G4
  bombus_pascuorum: orthologs found for 117/120 candidates
  bombus_vancouverensis: orthologs found for 118/120 candidates
  bombus_huntii: orthologs found for 118/120 candidates
  bombus_vosnesenskii: orthologs found for 118/120 candidates
  bombus_fervidus: orthologs found for 118/120 candidates
  bombus_affinis: orthologs found for 118/120 candidates
  bombus_pyrosoma: orthologs found for 119/120 candidates
promoter-G4 recovered per organism: {'bombus_terrestris': 120, 'bombus_huntii': 59, 'bombus_affinis': 87, 'bombus_vancouverensis': 64, 'bombus_pascuorum': 59, 'bombus_fervidus': 53, 'bombus_pyrosoma': 56, 'bombus_vosnesenskii': 55}
100 gene(s) have a promoter G4 in >=2 organisms

=== promoter-G4 ortholog #1  (ref protein XP_048260124.1) ===
  bombus_terrestris    len= 20 tracts=4 [3, 5, 5, 5] loops=[1, 1]  [NC_063269.1:199259-199279 + -> intron]
  bombus_huntii        len= 26 tracts=5 [4, 5, 5, 5, 3] loops=[4]  [NC_066246.

## Гены с Z-ДНК в промотерах

Делаем то же самое, но выравниваем последовательности промотеров полностью.

In [ ]:
def alt_run(seq):
    best = cur = 1
    for i in range(1, len(seq)):
        if seq[i].upper() != seq[i-1].upper() and {seq[i].upper(), seq[i-1].upper()} <= set('ACGT'):
            cur += 1; best = max(best, cur)
        else: cur = 1
    return best

omapz = orthologous_promoter_genes('zdna', 'Zdna')

per_orgz, allprom = Counter(), {}
for p, m in omapz.items():
    proms = {}
    for org, prot in m.items():
        prom, zd = promoter_structures(org, prot, 'zdna')
        if zd and prom: proms[org] = prom; per_orgz[org] += 1
    allprom[p] = proms
print('promoter Z-DNA recovered per organism:', dict(per_orgz))

sharedz = [(p, s) for p, s in allprom.items() if len(s) >= 2]
print(f'{len(sharedz)} gene(s) have promoter Z-DNA in >=2 organisms')
for i, (p, proms) in enumerate(sharedz[:9], 1):
    print(f'\n=== promoter-Z-DNA ortholog #{i}  (ref protein {p}) ===')
    for org, s in proms.items():
        print(f'  {org:20s} promoter_len={len(s)} longest_alt_run={alt_run(s)}')
    faa, aln = f'group/seqs/promZ_{i}.fa', f'group/aln/promZ_{i}.aln'
    with open(faa, 'w') as fh:
        for org, s in proms.items(): fh.write(f'>{org}\n{s}\n')
    mafft(faa, aln)
    print(f'  promoters aligned -> {aln}')
print('\npromoter Z-DNA alignments -> group/aln/promZ_*.aln')

Чтобы подсветить в нашем выравнивании Z-ДНК пишем хэлпер.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgba
from Bio import SeqIO, AlignIO

def zdna_positions_in_promoter(org, protein_id, flank=FLANK):
    loc = gene_locus(org, protein_id)
    if not loc:
        return []
    pline = promoter_bed(org, loc)
    if not pline:
        return []

    f = pline.rstrip('\n').split('\t')
    prom_chrom, prom_s, prom_e = f[0], int(f[1]), int(f[2])
    prom_strand = f[5] if len(f) >= 6 else '+'

    with open('group/promoters/_prom.bed', 'w') as fh:
        fh.write(pline)

    zdna_bed = ORGANISMS[org].get('zdna')
    inter = _safe_intersect(zdna_bed, 'group/promoters/_prom.bed', '-wo')
    if not inter:
        return []

    regions = []
    prom_len = prom_e - prom_s
    for line in inter:
        cols = line.rstrip('\n').split('\t')
        zs, ze = int(cols[1]), int(cols[2])
        cs, ce = max(zs, prom_s), min(ze, prom_e)
        if ce <= cs:
            continue
        if prom_strand == '-':
            rel_s = prom_e - ce
            rel_e = prom_e - cs
        else:
            rel_s = cs - prom_s
            rel_e = ce - prom_s
        regions.append((rel_s, rel_e))
    return regions

Дальше пишем функции для красивого выравнивания

In [ ]:
def plot_alignment_compact(aln_path, annotations_per_seq=None,
                           ann_color='#8e44ad', ann_alpha=0.5,
                           title='', out_path=None,
                           show_conservation=True):
    aln     = AlignIO.read(aln_path, 'fasta')
    records = list(aln)
    n_seq   = len(records)
    aln_len = aln.get_alignment_length()
    seqs = np.array([list(str(r.seq).upper()) for r in records])
    gap_mask  = seqs == '-'
    consensus = np.array([
        max(set(col[col != '-']), key=lambda c: np.sum(col == c))
        if np.any(col != '-') else '-'
        for col in seqs.T
    ])
    palette = np.array([[52/255, 152/255, 219/255],
                        [230/255, 126/255, 34/255],
                        [1.0, 1.0, 1.0]])

    img = np.zeros((n_seq, aln_len), dtype=int)
    for j in range(aln_len):
        for i in range(n_seq):
            if seqs[i, j] == '-':
                img[i, j] = 2
            elif seqs[i, j] == consensus[j]:
                img[i, j] = 0
            else:
                img[i, j] = 1

    rgb = palette[img]
    pct_id = np.array([
        np.sum(seqs[:, j] == consensus[j]) / n_seq
        for j in range(aln_len)
    ])
    label_len = max(len(r.id.replace('bombus_', 'B. ')) for r in records)
    fig_w = max(14, aln_len / 60)
    row_h = 0.45
    cons_h = 1.2
    if show_conservation:
        fig, (ax_cons, ax_aln) = plt.subplots(
            2, 1, figsize=(fig_w, cons_h + n_seq * row_h + 0.8),
            gridspec_kw={'height_ratios': [cons_h, n_seq * row_h],
                         'hspace': 0.04})
    else:
        fig, ax_aln = plt.subplots(figsize=(fig_w, n_seq * row_h + 0.8))
        ax_cons = None

    if ax_cons is not None:
        ax_cons.fill_between(np.arange(aln_len), pct_id,
                             color='#27ae60', linewidth=0, alpha=0.9)
        ax_cons.set_xlim(0, aln_len)
        ax_cons.set_ylim(0, 1.05)
        ax_cons.set_yticks([0, 0.5, 1])
        ax_cons.set_yticklabels(['0', '.5', '1'], fontsize=7)
        ax_cons.set_ylabel('% id', fontsize=8)
        ax_cons.set_xticks([])
        ax_cons.spines[['top', 'right', 'bottom']].set_visible(False)
        ax_cons.set_title(title, fontsize=10, fontweight='bold', pad=6)
    ax_aln.imshow(rgb, aspect='auto', interpolation='nearest',
                  extent=[0, aln_len, 0, n_seq])
    ax_aln.set_yticks(np.arange(n_seq) + 0.5)
    ax_aln.set_yticklabels(
        [r.id.replace('bombus_', 'B. ') for r in reversed(records)],
        fontsize=8, style='italic')
    tick_step = max(50, aln_len // 6)
    ticks     = list(range(0, aln_len, tick_step)) + [aln_len]
    ax_aln.set_xticks(ticks)
    ax_aln.set_xticklabels([t + 1 for t in ticks], fontsize=7)
    ax_aln.set_xlabel(f'promoter alignment position (1–{aln_len})', fontsize=8)
    ax_aln.set_xlim(0, aln_len)
    if annotations_per_seq:
        def seq_to_aln_map(aligned_seq):
            mapping, orig = {}, 0
            for col, ch in enumerate(aligned_seq):
                if ch != '-':
                    mapping[orig] = col
                    orig += 1
            return mapping

        s2a_maps = {r.id: seq_to_aln_map(str(r.seq)) for r in records}

        for row_i, rec in enumerate(records):
            if rec.id not in annotations_per_seq:
                continue
            y_bot = n_seq - row_i - 1
            s2a   = s2a_maps[rec.id]

            for (orig_s, orig_e) in annotations_per_seq[rec.id]:
                aln_s = s2a.get(orig_s)
                aln_e = s2a.get(orig_e - 1)
                if aln_s is None or aln_e is None:
                    continue
                ax_aln.add_patch(mpatches.Rectangle(
                    (aln_s, y_bot), aln_e - aln_s + 1, 1.0,
                    linewidth=1.2, edgecolor=ann_color,
                    facecolor=to_rgba(ann_color, ann_alpha),
                    zorder=3))

    ax_aln.spines[['top', 'right']].set_visible(False)

    if not show_conservation:
        ax_aln.set_title(title, fontsize=10, fontweight='bold', pad=6)
    legend_patches = [
        mpatches.Patch(color='#3498db', label='conserved'),
        mpatches.Patch(color='#e67e22', label='variant'),
        mpatches.Patch(facecolor='white', edgecolor='gray',
                       linewidth=0.8, label='gap'),
    ]
    if annotations_per_seq:
        legend_patches.append(mpatches.Patch(
            facecolor=to_rgba(ann_color, ann_alpha),
            edgecolor=ann_color, linewidth=1.2, label='Z-DNA'))
    ax_aln.legend(handles=legend_patches, loc='lower center',
                  bbox_to_anchor=(0.5, -0.28), ncol=len(legend_patches),
                  fontsize=8, framealpha=0.9)

    plt.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=180, bbox_inches='tight')
        print(f'  -> saved: {out_path}')
    plt.show()
    plt.close()

In [ ]:
import os
os.makedirs('group/figures/zdna_aln', exist_ok=True)

for i, (prot_ref, proms) in enumerate(sharedz[:9], 1):
    aln_path = f'group/aln/promZ_{i}.aln'
    if not os.path.exists(aln_path):
        print(f'[skip] {aln_path} not found')
        continue
    ann = {}
    for org, prot_id in omapz[prot_ref].items():
        if org not in proms:
            continue
        zdna_regs = zdna_positions_in_promoter(org, prot_id)
        if zdna_regs:
            ann[org] = zdna_regs  # ключи = record.id в .aln файле

    title = f'Promoter Z-DNA alignment #{i}  ({prot_ref})\nZ-DNA regions highlighted in purple'
    out = f'group/figures/zdna_aln/promZ_{i}_annotated.png'

    print(f'\n=== Z-DNA alignment #{i} ({prot_ref}) ===')
    print(f'  Z-DNA annotations found for: {list(ann.keys()) or "none"}')
    plot_alignment_compact(aln_path, annotations_per_seq=ann,
                       ann_color='#8e44ad', ann_alpha=0.45,
                       title=f'Promoter Z-DNA alignment #{i}  ({prot_ref})',
                       out_path=out)
    print(f'  -> saved: {out}')

И отдельно выравняем и нарисуем фрагменты с наибольшим потенциалом формирования Z-ДНК.

In [ ]:
NT_COLORS = {
    'A': '#2ecc71',  # зелёный
    'T': '#e74c3c',  # красный
    'G': '#f39c12',  # оранжевый
    'C': '#3498db',  # синий
    '-': '#ecf0f1',  # светло-серый (гэп)
    'N': '#bdc3c7',
}

def plot_alignment_annotated(aln_path, annotations_per_seq=None,
                             ann_color='#9b59b6', ann_alpha=0.35,
                             title='', max_cols=200, out_path=None):
    aln = AlignIO.read(aln_path, 'fasta')
    records = list(aln)
    n_seq = len(records)
    aln_len = aln.get_alignment_length()
    show_len = min(aln_len, max_cols)
    def seq_to_aln_map(aligned_seq):
        mapping = {}
        orig = 0
        for col, ch in enumerate(str(aligned_seq)):
            if ch != '-':
                mapping[orig] = col
                orig += 1
        return mapping

    fig_w = max(14, show_len * 0.085)
    fig_h = max(3, n_seq * 0.55 + 1.2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    cell_w, cell_h = 1.0, 1.0

    for row_i, rec in enumerate(records):
        seq = str(rec.seq)[:show_len]
        s2a = seq_to_aln_map(str(rec.seq))
        y = (n_seq - row_i - 1) * cell_h
        for col_i, ch in enumerate(seq):
            color = NT_COLORS.get(ch.upper(), '#bdc3c7')
            ax.add_patch(mpatches.FancyBboxPatch(
                (col_i * cell_w, y), cell_w, cell_h,
                boxstyle='square,pad=0', linewidth=0,
                facecolor=color, zorder=1))
            if show_len <= 120:
                ax.text(col_i * cell_w + 0.5, y + 0.5, ch.upper(),
                        ha='center', va='center',
                        fontsize=max(4, 8 - show_len // 30),
                        fontweight='bold', color='white', zorder=3)
        if annotations_per_seq and rec.id in annotations_per_seq:
            for (orig_s, orig_e) in annotations_per_seq[rec.id]:
                aln_s = s2a.get(orig_s)
                aln_e = s2a.get(orig_e - 1)
                if aln_s is None or aln_e is None:
                    continue
                aln_s = min(aln_s, show_len - 1)
                aln_e = min(aln_e + 1, show_len)
                if aln_s >= aln_e:
                    continue
                ax.add_patch(mpatches.FancyBboxPatch(
                    (aln_s * cell_w, y - 0.05),
                    (aln_e - aln_s) * cell_w, cell_h + 0.1,
                    boxstyle='square,pad=0',
                    linewidth=1.5, edgecolor=ann_color,
                    facecolor=to_rgba(ann_color, ann_alpha),
                    zorder=2))
        ax.text(-0.5, y + 0.5, rec.id.replace('bombus_', 'B.'),
                ha='right', va='center', fontsize=8, style='italic')

    ax.set_xlim(-len(max([r.id for r in records], key=len)) * 0.12, show_len * cell_w)
    ax.set_ylim(-0.3, n_seq * cell_h + 0.5)
    ax.set_xticks(np.arange(0, show_len + 1, max(1, show_len // 10)) + 0.5)
    ax.set_xticklabels(range(0, show_len + 1, max(1, show_len // 10)), fontsize=7)
    ax.set_yticks([])
    ax.set_xlabel('Alignment column', fontsize=9)
    nt_legend = [mpatches.Patch(color=c, label=nt)
                 for nt, c in NT_COLORS.items() if nt != 'N']
    ann_patch = mpatches.Patch(facecolor=to_rgba(ann_color, ann_alpha),
                                edgecolor=ann_color, linewidth=1.5, label='Z-DNA region')
    ax.legend(handles=nt_legend + [ann_patch],
              loc='upper right', ncol=len(nt_legend) + 1,
              fontsize=7, framealpha=0.9)

    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
    plt.tight_layout()

    if out_path:
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
def get_all_ry_runs(seq, min_len=6):
    pattern = re.compile(r'(?:[AG][CT])+|(?:[CT][AG])+', re.IGNORECASE)
    runs = []
    for m in pattern.finditer(seq):
        if len(m.group()) >= min_len:
            runs.append((m.start(), m.end(), m.group()))
    return runs


os.makedirs('group/figures/ry_aln', exist_ok=True)
os.makedirs('group/seqs', exist_ok=True)
os.makedirs('group/aln', exist_ok=True)

for i, (prot_ref, proms) in enumerate(sharedz[:9], 1):
    print(f'\n=== RY-run alignment #{i} (ref: {prot_ref}) ===')
    ry_seqs   = {}
    ry_coords = {}

    for org, prom_seq in proms.items():
        runs = get_all_ry_runs(prom_seq, min_len=6)
        if not runs:
            continue
        best = max(runs, key=lambda r: r[1] - r[0])
        ry_seqs[org]   = best[2]
        ry_coords[org] = (best[0], best[1])
        print(f'  {org:25s} longest_RY={best[1]-best[0]:3d} bp  '
              f'at [{best[0]}:{best[1]}]  seq: {best[2][:40]}{"..." if len(best[2])>40 else ""}')

    if len(ry_seqs) < 2:
        print('  Not enough organisms with RY runs — skipping alignment.')
        continue
    faa = f'group/seqs/promZ_RY_{i}.fa'
    aln = f'group/aln/promZ_RY_{i}.aln'
    with open(faa, 'w') as fh:
        for org, s in ry_seqs.items():
            fh.write(f'>{org}\n{s}\n')
    mafft(faa, aln)
    ann_ry = {org: [(0, len(seq))] for org, seq in ry_seqs.items()}

    title = (f'Longest RY-run alignment #{i}  ({prot_ref})\n'
             f'RY alternating region highlighted in orange')
    out = f'group/figures/ry_aln/promZ_RY_{i}_annotated.png'

    plot_alignment_annotated(aln, annotations_per_seq=ann_ry,
                             ann_color='#e67e22', ann_alpha=0.35,
                             title=title, max_cols=200, out_path=out)
    print(f'  -> saved: {out}')
    ref_org = list(proms.keys())[0]
    ref_seq = proms[ref_org]
    all_runs = get_all_ry_runs(ref_seq, min_len=6)
    if all_runs:
        print(f'\n  All RY runs in {ref_org} (min_len=6):')
        for s, e, seq in sorted(all_runs, key=lambda r: -(r[1]-r[0]))[:10]:
            print(f'    [{s:4d}:{e:4d}] len={e-s:3d}  {seq[:50]}')

Далее пришлось немножко извратиться, чтобы найти названия генов по найденным участкам. Для этого мы ходим в uniprot по API.

In [ ]:
from urllib.parse import unquote
import pandas as pd

def _ref_annot(org):
    k = 'annot_' + org
    if k in _CACHE: return _CACHE[k]
    gene_name, rna_prod, prod = {}, {}, {}
    with open(ORGANISMS[org]['gff']) as fh:
        for line in fh:
            if line.startswith('#'): continue
            f = line.rstrip('\n').split('\t')
            if len(f) < 9: continue
            a = dict(x.split('=', 1) for x in f[8].split(';') if '=' in x)
            if f[2] == 'gene': gene_name[a.get('ID')] = a.get('Name') or a.get('gene') or a.get('ID')
            elif f[2] in ('mRNA', 'transcript') and 'ID' in a: rna_prod[a['ID']] = unquote(a.get('product', ''))
            elif f[2] == 'CDS' and 'protein_id' in a:
                prod.setdefault(a['protein_id'], unquote(a.get('product', '')) or rna_prod.get(a.get('Parent'), ''))
    _CACHE[k] = (gene_name, prod); return _CACHE[k]

def _swissprot_names(ref_prots, org):
    SP = 'group/db/sprot.fasta'
    try:
        if not os.path.exists('group/db/sprot.dmnd'):
            if not os.path.exists(SP):
                url = ('https://ftp.uniprot.org/pub/databases/uniprot/current_release/'
                       'knowledgebase/complete/uniprot_sprot.fasta.gz')
                sh(f'wget -q -O {SP}.gz "{url}" && gunzip -f {SP}.gz')
            sh(f'diamond makedb --in {SP} -d group/db/sprot --quiet')
        qf, idx = 'group/seqs/_namequery.faa', proteome(org)
        with open(qf, 'w') as fh:
            for p in dict.fromkeys(ref_prots):
                if p in idx: fh.write(f'>{p}\n{idx[p].seq}\n')
        out = 'group/blast/sprot_names.tsv'
        sh(f'diamond blastp -q {qf} -d group/db/sprot -o {out} --very-sensitive '
           f'--max-target-seqs 1 --evalue 1e-3 --threads 4 --outfmt 6 qseqid stitle pident evalue --quiet')
        best = {}
        for line in open(out):
            q, st, pid, ev = line.rstrip('\n').split('\t')
            if q in best: continue
            desc = st.split(' ', 1)[1] if ' ' in st else st
            name = desc.split(' OS=')[0]
            gn = st.split('GN=')[1].split()[0] if 'GN=' in st else ''
            sp = st.split('OS=')[1].split(' OX=')[0] if 'OS=' in st else ''
            best[q] = (gn, name, sp, float(pid), float(ev))
        return best
    except Exception as e:
        print('  SwissProt naming skipped:', e); return {}

REF_ORG = next(iter(ORGANISMS))
gene_name, prod = _ref_annot(REF_ORG)
_, _, _, prot2gene = _gff_maps(REF_ORG)

g4_ref = [p for p, _ in shared]  if 'shared'  in globals() else []
z_ref  = [p for p, _ in sharedz] if 'sharedz' in globals() else []
assert g4_ref or z_ref, 'run Parts C and D first (need `shared` / `sharedz`)'
sp = _swissprot_names(g4_ref + z_ref, REF_ORG)

def _rows(ref_list, prefix):
    out = []
    for i, p in enumerate(ref_list, 1):
        g = prot2gene.get(p)
        gn, nm, spc, pid, ev = sp.get(p, ('', '', '', None, None))
        out.append({'alignment': f'{prefix}_{i}', 'ref_protein': p,
                    'gene_id': g or '', 'gff_symbol': gene_name.get(g, '') if g else '',
                    'gff_product': prod.get(p, ''),
                    'homolog_gene': gn, 'homolog_name': nm, 'homolog_species': spc,
                    'pident': f'{pid:.0f}' if pid is not None else '',
                    'evalue': f'{ev:.0e}' if ev else ''})
    return out

g4_genes = pd.DataFrame(_rows(g4_ref, 'promG4'))
z_genes  = pd.DataFrame(_rows(z_ref,  'promZ'))
g4_genes.to_csv('group/promG4_genes.csv', index=False)
z_genes.to_csv('group/promZ_genes.csv', index=False)
print('=== promoter-G4 alignment genes ===');    print(g4_genes.to_string(index=False))
print('\n=== promoter-Z-DNA alignment genes ==='); print(z_genes.to_string(index=False))
print('\nSaved group/promG4_genes.csv and group/promZ_genes.csv')


## Подсчет итоговых статистик

In [ ]:
import pandas as pd
def _nl(f):
    return sum(1 for _ in open(f)) if f and os.path.exists(f) else 0

def build_features(org):
    d = f'group/feat/{org}'; os.makedirs(d, exist_ok=True)
    cs = chrom_sizes(org); sizes = f'{d}/genome.sizes'
    with open(sizes, 'w') as fh:
        for c, l in cs.items(): fh.write(f'{c}\t{l}\n')
    sh(f'sort -k1,1 {sizes} -o {sizes}')

    genes, exons = [], []
    with open(ORGANISMS[org]['gff']) as fh:
        for line in fh:
            if line.startswith('#'): continue
            f = line.rstrip('\n').split('\t')
            if len(f) < 9 or f[0] not in cs: continue
            if   f[2] == 'gene': genes.append((f[0], int(f[3])-1, int(f[4]), f[6]))
            elif f[2] == 'exon': exons.append((f[0], int(f[3])-1, int(f[4])))

    g = f'{d}/genes.bed'
    with open(g, 'w') as fh:
        for c, s, e, st in genes: fh.write(f'{c}\t{s}\t{e}\tg\t0\t{st}\n')
    sh(f'sort -k1,1 -k2,2n {g} -o {g}')
    e_bed = f'{d}/exons.bed'
    with open(f'{d}/_ex.bed', 'w') as fh:
        for c, s, en in exons: fh.write(f'{c}\t{s}\t{en}\n')
    sh(f'bedtools sort -i {d}/_ex.bed | bedtools merge -i stdin > {e_bed}')
    intr = f'{d}/introns.bed'
    sh(f'bedtools subtract -a {g} -b {e_bed} | sort -k1,1 -k2,2n | bedtools merge -i stdin > {intr}')
    prom = f'{d}/promoters.bed'
    with open(prom, 'w') as fh:
        for c, s, en, st in genes:
            ps, pe = (max(0, s-PROM), s) if st == '+' else (en, min(cs[c], en+PROM))
            if pe > ps: fh.write(f'{c}\t{ps}\t{pe}\n')
    sh(f'bedtools sort -i {prom} | bedtools merge -i stdin > {prom}.t && mv {prom}.t {prom}')
    inter = f'{d}/intergenic.bed'
    sh(f'bedtools complement -i {g} -g {sizes} | bedtools subtract -a stdin -b {prom} > {inter}')
    return {'genes': g, 'exons': e_bed, 'introns': intr, 'promoters': prom, 'intergenic': inter}, len(genes)

FEAT_ORDER = ['genes', 'exons', 'introns', 'promoters', 'intergenic']

summ_rows, g4_rows, z_rows = [], [], []
for org, P in ORGANISMS.items():
    if not (P.get('genome') and os.path.exists(P['genome'])
            and P.get('gff') and os.path.exists(P['gff'])):
        print(f'skip {org}: genome/gff missing'); continue
    try:
        feats, n_genes = build_features(org)
    except Exception as ex:
        print(f'skip {org}: feature build failed ({ex})'); continue

    cs = chrom_sizes(org)
    n_g4, n_z = _nl(P.get('g4')), _nl(P.get('zdna'))
    name = org.replace('_', ' ').title()
    summ_rows.append({
        'organism': name,
        'assembly / DB id': P.get('assembly') or f'{next(iter(cs))} … ({len(cs)} seqs)',
        'length, bp': sum(cs.values()),
        '# sequences': len(cs),
        '# genes': n_genes,
        '# G4': n_g4,
        '# Z-DNA': n_z,
    })
    g4r = {'organism': name, 'total': n_g4}
    zr  = {'organism': name, 'total': n_z}
    for feat in FEAT_ORDER:
        g4r[feat] = len(_safe_intersect(P.get('g4'),   feats[feat], '-u')) if n_g4 else 0
        zr[feat]  = len(_safe_intersect(P.get('zdna'), feats[feat], '-u')) if n_z  else 0
    g4_rows.append(g4r); z_rows.append(zr)

summary = pd.DataFrame(summ_rows).set_index('organism')
g4_dist = pd.DataFrame(g4_rows).set_index('organism')
z_dist  = pd.DataFrame(z_rows).set_index('organism')
summary.to_csv('group/summary_genomes.csv')
g4_dist.to_csv('group/summary_g4_distribution.csv')
z_dist.to_csv('group/summary_zdna_distribution.csv')

print('=== Genome summary (group/summary_genomes.csv) ===')
print(summary.to_string())
print('\n=== G4 distribution — # G4 overlapping each feature (group/summary_g4_distribution.csv) ===')
print(g4_dist.to_string())
print('\n=== Z-DNA distribution — # Z-DNA overlapping each feature (group/summary_zdna_distribution.csv) ===')
print(z_dist.to_string())


### Таблица используемых параметров запуска

| Step | Tool | Parameters |
|---|---|---|
| Epigenetic / CTCF-cohesin | **DIAMOND** `blastp` | `--very-sensitive --max-target-seqs 1 --evalue 1e-5` |
| Protein & DNA alignments | **MAFFT** | `--auto` |
| Gene trees | **FastTree** | белки `-lg`, нуклеотиды `-nt -gtr` |
| G-quadruplexes | regex (Python) | `(?:G{3,5}[ATGC]{1,7}){3,}G{3,5}`, оба стренда, без учета регистра |
| Z-DNA | **Z-Hunt** *or* **ZDNABERT** | Z-Hunt `zhunt 12 8 12`, z-score > 400 · ZDNABERT prob > 0.5, min len 10 bp |
| Promoters / features | GFF3 + bedtools | promoter = 1000 bp upstream от TSS |